# INJECTION-R4: Final Joint Polish

**Stage:** R-4 — final joint polish across all three heads  
**Input:** `r3_best_weights.h5` (from `model/unified_model/new_unified_model/`)  
**Output:** `r4_final_weights.h5` + `r4_final_eval_report.json`

### What this stage does
- Loads R3 weights and **unfreezes everything** (encoder + all heads)
- Joint alternating training: ATS+Domain batch → RSG batch, canonical loss weights
- Very low LR (`1e-5`) — polish only, not retraining
- Early stopping on `val_loss`, patience = 8

### Final Hard Gates — ALL must pass
| Metric | Gate |
|--------|------|
| ATS val MAE (0–100) | < 6.5 |
| ATS test MAE (0–100) | < 7.0 |
| Domain F1 macro | > 0.85 |
| Domain F1 per-class (all 7) | > 0.80 each |
| RSG val Accuracy | > 0.65 |
| Fresher fairness gap | ≤ 20 pts |

### Required files — upload to `ATS_R4/` on Google Drive
```
MyDrive/ATS_R4/
  ats_tokenized.npz          <- from data/tokenized/
  rsg_tokenized.npz          <- from data/tokenized/
  data_splits.json           <- from data/tokenized/
  r3_best_weights.h5         <- from model/unified_model/new_unified_model/
  merged_final.csv           <- from data/labeled/  (optional: fresher fairness)
```
Output (`r4_final_weights.h5`, `r4_final_eval_report.json`) auto-saved to `ATS_R4/output/`.

> **Runtime:** Set to **T4 GPU** via `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Cell 1: GPU check
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('No GPU detected!')
    print('Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU')
    print('Then re-run all cells.')
else:
    print(result.stdout)

In [ ]:
# Cell 2: Install dependencies
# Pin transformers to 4.44.2 — TFAutoModel was removed in 4.47+
!pip install -q "transformers==4.44.2" scikit-learn
print('Dependencies installed.')
print()
print('>>> ACTION REQUIRED: Runtime -> Restart session, then re-run ALL cells from Cell 1. <<<')

In [ ]:
# Cell 3: Mount Google Drive & configure paths
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
#  CONFIGURE: set this to your ATS_R4 folder on Drive
# ================================================================
DRIVE_ROOT = '/content/drive/MyDrive/ATS_R4'

import os
from pathlib import Path

DATA_DIR        = Path(DRIVE_ROOT)
R3_WEIGHTS      = str(DATA_DIR / 'r3_best_weights.h5')
SPLITS_JSON     = str(DATA_DIR / 'data_splits.json')
MERGED_CSV      = str(DATA_DIR / 'merged_final.csv')   # optional — for fresher fairness

OUT_DIR         = DATA_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)
R4_BEST         = str(OUT_DIR / 'r4_final_weights.h5')
R4_LOG          = str(OUT_DIR / 'r4_training_log.csv')
R4_REPORT       = str(OUT_DIR / 'r4_final_eval_report.json')
R4_CKPT         = str(OUT_DIR / 'r4_best_in_progress.weights.h5')

print(f'Drive root  : {DRIVE_ROOT}')
print(f'R3 weights  : {R3_WEIGHTS}')
print(f'Splits      : {SPLITS_JSON}')
print(f'R4 output   : {OUT_DIR}')

In [ ]:
# Cell 4: Verify required files
required = [
    str(DATA_DIR / 'ats_tokenized.npz'),
    str(DATA_DIR / 'rsg_tokenized.npz'),
    R3_WEIGHTS,
    SPLITS_JSON,
]
optional = [MERGED_CSV]

all_ok = True
for f in required:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f) / 1e6:.1f} MB' if exists else 'MISSING'
    status = 'OK     ' if exists else 'MISSING'
    print(f'  [{status}]  {size:>10}   {f}')
    all_ok = all_ok and exists

for f in optional:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f) / 1e6:.1f} MB' if exists else 'not uploaded'
    print(f'  [OPTIONAL]  {size:>10}   {f}')

if not all_ok:
    raise FileNotFoundError('One or more required files are missing. Upload them to Drive first.')
print('\nAll required files found -- ready to train.')

In [ ]:
# Cell 5: Imports & environment
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import csv, json
import numpy as np
import tensorflow as tf
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')
from sklearn.metrics import f1_score, mean_absolute_error

print(f'TensorFlow : {tf.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Cell 6: Hyperparameters
MINILM_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
SEQ_LEN           = 128

# R4 training config (canonical, per spec)
LR            = 1e-5      # very low — polish only
BATCH_SIZE    = 32
MAX_EPOCHS    = 30
PATIENCE      = 8

# Canonical loss weights — MUST NOT change
W_ATS = 0.35
W_DOM = 0.35
W_RSG = 0.30
assert round(W_ATS + W_DOM + W_RSG, 4) == 1.0, 'Loss weights must sum to 1.0'

# Domain class weights (from config.py)
DOMAIN_CLASS_WEIGHTS = {0: 1.4, 1: 0.8, 2: 0.9, 3: 1.0, 4: 1.5, 5: 0.9, 6: 1.0}
DOMAIN_NAMES = ['IT', 'Non-IT', 'Design', 'Healthcare', 'Finance', 'Legal', 'Edu']

# Hard gates (spec)
GATE_ATS_VAL_MAE  = 6.5    # ATS val MAE (0-100) must be < 6.5
GATE_ATS_TEST_MAE = 7.0    # ATS test MAE (0-100) must be < 7.0
GATE_DOM_F1_MACRO = 0.85   # Domain F1 macro must be > 0.85
GATE_DOM_F1_CLASS = 0.80   # Each of 7 domain F1 must be > 0.80
GATE_RSG_ACC      = 0.65   # RSG val accuracy must be > 0.65
GATE_FRESHER_GAP  = 20.0   # Fresher fairness gap must be <= 20 pts

# Hard-stop threshold during training (regression guard)
HARD_STOP_MAE_100 = 8.5    # abort epoch if val ATS MAE exceeds this

# Fresher keywords for fairness check
FRESHER_KEYWORDS = [
    'fresher', 'entry level', 'entry-level', '0 years experience',
    '0-1 years', 'recent graduate', 'final year', 'fresh graduate',
    'no prior experience', 'no experience',
]

print('Hyperparameters configured.')
print(f'  LR={LR}  BATCH={BATCH_SIZE}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}')
print(f'  Loss weights: ATS={W_ATS}  DOM={W_DOM}  RSG={W_RSG}  (canonical)')
print(f'  Hard-stop ceiling: ATS MAE <= {HARD_STOP_MAE_100}')

In [ ]:
# Cell 7: MiniLM model definition (R4 — encoder trainable)
import transformers as _tf_check
_major, _minor = (int(x) for x in _tf_check.__version__.split('.')[:2])
if (_major, _minor) >= (4, 47):
    raise RuntimeError(
        f'transformers {_tf_check.__version__} is loaded — TFAutoModel was removed in 4.47.\n'
        'Fix: run Cell 2, then Runtime -> Restart session, then re-run all cells from Cell 1.'
    )

from transformers import TFAutoModel


def mean_pool_l2(last_hidden, attention_mask):
    mask    = tf.cast(tf.expand_dims(attention_mask, -1), tf.float32)
    sum_emb = tf.reduce_sum(last_hidden * mask, axis=1)
    count   = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-9)
    return tf.nn.l2_normalize(sum_emb / count, axis=-1)


class MiniLMEncoderLayer(tf.keras.layers.Layer):
    """MiniLM encoder layer — R4: training mode propagates to BERT for fine-tuning."""
    def __init__(self, bert_model, **kwargs):
        super().__init__(**kwargs)
        self._bert = bert_model

    def call(self, inputs, training=False):
        input_ids, attention_mask = inputs
        out = self._bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=tf.zeros_like(input_ids),
            training=training,   # propagated so encoder fine-tunes in R4
        )
        return mean_pool_l2(out.last_hidden_state, attention_mask)

    def get_config(self):
        return super().get_config()


def build_unified_minilm_model(bert_model):
    resume_ids  = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_input_ids')
    resume_mask = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_attention_mask')
    jd_ids      = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_input_ids')
    jd_mask     = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_attention_mask')

    encoder    = MiniLMEncoderLayer(bert_model, name='minilm_encoder')
    resume_emb = encoder([resume_ids,  resume_mask])
    jd_emb     = encoder([jd_ids,      jd_mask])

    cosine_sim = tf.keras.layers.Dot(axes=1, normalize=True,  name='cosine_sim')([resume_emb, jd_emb])
    dot_prod   = tf.keras.layers.Dot(axes=1, normalize=False, name='dot_prod')  ([resume_emb, jd_emb])
    ats_feat   = tf.keras.layers.Concatenate(name='ats_features')([resume_emb, jd_emb, cosine_sim, dot_prod])

    x1        = tf.keras.layers.Dense(256, activation='relu',   name='ats_dense1')(ats_feat)
    x1        = tf.keras.layers.Dropout(0.3,                    name='ats_drop1')(x1)
    x1        = tf.keras.layers.Dense(64,  activation='relu',   name='ats_dense2')(x1)
    x1        = tf.keras.layers.Dropout(0.2,                    name='ats_drop2')(x1)
    ats_score = tf.keras.layers.Dense(1, activation='sigmoid',  name='ats_score')(x1)

    x2           = tf.keras.layers.Dense(256, activation='relu',  name='dom_dense1')(jd_emb)
    x2           = tf.keras.layers.Dropout(0.3,                   name='dom_drop1')(x2)
    x2           = tf.keras.layers.Dense(128, activation='relu',  name='dom_dense2')(x2)
    x2           = tf.keras.layers.Dropout(0.2,                   name='dom_drop2')(x2)
    domain_probs = tf.keras.layers.Dense(7, activation='softmax', name='domain_probs')(x2)

    x3           = tf.keras.layers.Dense(512, activation='relu',  name='rsg_dense1')(resume_emb)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn1')(x3)
    x3           = tf.keras.layers.Dropout(0.4,                   name='rsg_drop1')(x3)
    x3           = tf.keras.layers.Dense(256, activation='relu',  name='rsg_dense2')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn2')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop2')(x3)
    x3           = tf.keras.layers.Dense(128, activation='relu',  name='rsg_dense3')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn3')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop3')(x3)
    rsg_template = tf.keras.layers.Dense(46, activation='softmax', name='rsg_template')(x3)

    return tf.keras.Model(
        inputs=[resume_ids, resume_mask, jd_ids, jd_mask],
        outputs=[ats_score, domain_probs, rsg_template],
        name='unified_ats_minilm',
    )


print(f'transformers {_tf_check.__version__} -- TFAutoModel available.')
print('Model definition ready (R4: encoder supports training=True).')

In [ ]:
# Cell 8: Load tokenized data & splits
print('Loading tokenized data...')
ats_data = np.load(str(DATA_DIR / 'ats_tokenized.npz'))
rsg_data = np.load(str(DATA_DIR / 'rsg_tokenized.npz'))

with open(SPLITS_JSON) as f:
    splits = json.load(f)

ats_tr_idx  = np.array(splits['ats_train'])
ats_vl_idx  = np.array(splits['ats_val'])
ats_tst_idx = np.array(splits['ats_test'])
rsg_tr_idx  = np.array(splits['rsg_train'])
rsg_vl_idx  = np.array(splits['rsg_val'])

ATS_KEYS = ('resume_input_ids', 'resume_attention_mask',
             'jd_input_ids', 'jd_attention_mask', 'ats_scores', 'domain_labels')
RSG_KEYS = ('profile_input_ids', 'profile_attention_mask', 'rsg_labels')

ats_tr  = {k: ats_data[k][ats_tr_idx]  for k in ATS_KEYS}
ats_vl  = {k: ats_data[k][ats_vl_idx]  for k in ATS_KEYS}
ats_tst = {k: ats_data[k][ats_tst_idx] for k in ATS_KEYS}
rsg_tr  = {k: rsg_data[k][rsg_tr_idx]  for k in RSG_KEYS}
rsg_vl  = {k: rsg_data[k][rsg_vl_idx]  for k in RSG_KEYS}

n_ats_tr  = len(ats_tr_idx)
n_rsg_tr  = len(rsg_tr_idx)

print(f'  ATS  train={n_ats_tr:,}  val={len(ats_vl_idx):,}  test={len(ats_tst_idx):,}')
print(f'  RSG  train={n_rsg_tr:,}  val={len(rsg_vl_idx):,}')

In [ ]:
# Cell 9: Build model & load R3 weights
print(f'Loading MiniLM encoder ({MINILM_MODEL_NAME})...')
bert_model = TFAutoModel.from_pretrained(MINILM_MODEL_NAME, from_pt=True)
# Keep trainable=True so R4 can unfreeze the encoder
bert_model.trainable = True

model = build_unified_minilm_model(bert_model)
model.load_weights(R3_WEIGHTS)
print(f'R3 weights loaded: {R3_WEIGHTS}')
print(f'Total params: {model.count_params():,}')

In [ ]:
# Cell 10: Unfreeze EVERYTHING (encoder + all heads)
for layer in model.layers:
    layer.trainable = True

# Also ensure the inner bert_model is trainable
bert_model.trainable = True

trainable_params = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
total_params     = model.count_params()

print('R4 freeze strategy: UNFREEZE EVERYTHING')
print(f'  Trainable params : {trainable_params:,}')
print(f'  Total params     : {total_params:,}')
print(f'  Trainable layers : {[l.name for l in model.layers if l.trainable]}')

In [ ]:
# Cell 11: Eval helpers
_sce     = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
_sce_rsg = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


def eval_ats(data):
    """Returns (mae_100, dom_f1_macro, dom_ce, per_class_f1)."""
    n = len(data['ats_scores'])
    ats_preds, dom_preds = [], []
    for s in range(0, n, BATCH_SIZE):
        e = min(s + BATCH_SIZE, n)
        ap, dp, _ = model(
            [data['resume_input_ids'][s:e], data['resume_attention_mask'][s:e],
             data['jd_input_ids'][s:e],     data['jd_attention_mask'][s:e]],
            training=False,
        )
        ats_preds.append(ap.numpy())
        dom_preds.append(dp.numpy())
    ats_preds = np.concatenate(ats_preds).squeeze(-1)
    dom_preds = np.concatenate(dom_preds)

    mae_100      = float(np.mean(np.abs(ats_preds - data['ats_scores']))) * 100.0
    dom_true     = data['domain_labels']
    dom_pred_cls = np.argmax(dom_preds, 1)
    dom_f1_macro = f1_score(dom_true, dom_pred_cls, average='macro', zero_division=0)
    per_cls_f1   = f1_score(dom_true, dom_pred_cls, average=None,
                            labels=list(range(7)), zero_division=0)
    dom_ce       = float(
        tf.keras.losses.sparse_categorical_crossentropy(
            dom_true.astype('int32'), dom_preds
        ).numpy().mean()
    )
    return mae_100, dom_f1_macro, dom_ce, per_cls_f1


def eval_rsg(data):
    """Returns (top1_acc, top3_acc, ce_loss)."""
    m = len(data['rsg_labels'])
    rsg_preds = []
    for s in range(0, m, BATCH_SIZE):
        e    = min(s + BATCH_SIZE, m)
        pids = data['profile_input_ids'][s:e]
        pmsk = data['profile_attention_mask'][s:e]
        _, _, rp = model([pids, pmsk, pids, pmsk], training=False)
        rsg_preds.append(rp.numpy())
    rsg_preds = np.concatenate(rsg_preds)   # (N, 46)
    labels    = data['rsg_labels']

    top1_acc  = float(np.mean(np.argmax(rsg_preds, 1) == labels))
    top3_preds = np.argsort(rsg_preds, axis=1)[:, -3:]
    top3_acc  = float(np.mean([labels[i] in top3_preds[i] for i in range(m)]))
    ce        = float(
        tf.keras.losses.sparse_categorical_crossentropy(
            labels.astype('int32'), rsg_preds
        ).numpy().mean()
    )
    return top1_acc, top3_acc, ce


def combined_val_loss(ats_mae_01, dom_ce, rsg_ce):
    return W_ATS * ats_mae_01 + W_DOM * dom_ce + W_RSG * rsg_ce


print('Eval helpers ready.')

In [ ]:
# Cell 12: R3 baseline — measure before any R4 gradient
print('Measuring R3 baseline (val set)...')
r3_val_mae, r3_dom_f1, r3_dom_ce, _ = eval_ats(ats_vl)
r3_rsg_acc, r3_rsg_top3, r3_rsg_ce  = eval_rsg(rsg_vl)

print(f'  R3 val ATS MAE  : {r3_val_mae:.2f}  (gate < {GATE_ATS_VAL_MAE})')
print(f'  R3 val Dom F1   : {r3_dom_f1:.4f}  (gate > {GATE_DOM_F1_MACRO})')
print(f'  R3 val RSG Acc  : {r3_rsg_acc:.4f}  (gate > {GATE_RSG_ACC})')
print(f'  R3 val RSG Top-3: {r3_rsg_top3:.4f}')

# Save initial best
best_val_loss = combined_val_loss(r3_val_mae / 100.0, r3_dom_ce, r3_rsg_ce)
model.save_weights(R4_CKPT)
print(f'\n  Initial combined val loss: {best_val_loss:.4f}  (checkpoint saved)')

In [ ]:
# Cell 13: Optimizer & train steps
optimizer  = tf.keras.optimizers.Adam(learning_rate=LR)
_mae_loss  = tf.keras.losses.MeanAbsoluteError()


def train_step_ats_dom(r_ids, r_mask, jd_ids, jd_mask, ats_y, dom_y, dom_w):
    with tf.GradientTape() as tape:
        ats_p, dom_p, _ = model(
            [r_ids, r_mask, jd_ids, jd_mask], training=True
        )
        ats_l = _mae_loss(tf.expand_dims(ats_y, 1), ats_p)
        dom_l = _sce(dom_y, dom_p, sample_weight=dom_w)
        total = W_ATS * ats_l + W_DOM * dom_l
    optimizer.apply_gradients(
        zip(tape.gradient(total, model.trainable_variables),
            model.trainable_variables)
    )
    return float(ats_l), float(dom_l)


def train_step_rsg(p_ids, p_mask, rsg_y):
    with tf.GradientTape() as tape:
        _, _, rsg_p = model([p_ids, p_mask, p_ids, p_mask], training=True)
        loss = W_RSG * _sce_rsg(rsg_y, rsg_p)
    optimizer.apply_gradients(
        zip(tape.gradient(loss, model.trainable_variables),
            model.trainable_variables)
    )
    return float(loss)


print(f'Optimizer: Adam(lr={LR})')
print('Train steps: ATS+Domain (interleaved with RSG per batch)')

In [ ]:
# Cell 14: Joint alternating training loop
print('=' * 65)
print(f'  INJECTION-R4 Training  (max={MAX_EPOCHS} epochs, patience={PATIENCE})')
print(f'  Loss: ATS={W_ATS}  DOM={W_DOM}  RSG={W_RSG}  |  LR={LR}')
print('=' * 65)

rng              = np.random.default_rng(42)
patience_counter = 0
best_epoch       = 0
log_rows         = []
hard_stop        = False

# Pre-compute per-sample domain class weights for ATS train set
ats_tr_dom_w = np.array(
    [DOMAIN_CLASS_WEIGHTS.get(int(d), 1.0) for d in ats_tr['domain_labels']],
    dtype='float32'
)

n_ats_batches = -(-n_ats_tr // BATCH_SIZE)
print(f'  ATS batches/epoch: {n_ats_batches}  |  RSG train size: {n_rsg_tr:,}\n')

for epoch in range(1, MAX_EPOCHS + 1):
    # Shuffle indices each epoch
    ats_perm = rng.permutation(n_ats_tr)
    rsg_perm = rng.permutation(n_rsg_tr)
    rsg_pool = list(rsg_perm) * (n_ats_batches // max(1, n_rsg_tr // BATCH_SIZE) + 2)
    rsg_cursor = 0

    ats_ls, dom_ls, rsg_ls = [], [], []

    for bi in range(n_ats_batches):
        # ATS + Domain batch
        idx_a = ats_perm[bi * BATCH_SIZE : (bi + 1) * BATCH_SIZE]
        al, dl = train_step_ats_dom(
            ats_tr['resume_input_ids'][idx_a],
            ats_tr['resume_attention_mask'][idx_a],
            ats_tr['jd_input_ids'][idx_a],
            ats_tr['jd_attention_mask'][idx_a],
            ats_tr['ats_scores'][idx_a].astype('float32'),
            ats_tr['domain_labels'][idx_a].astype('int32'),
            ats_tr_dom_w[idx_a],
        )
        ats_ls.append(al)
        dom_ls.append(dl)

        # RSG batch (interleaved — wrap-around with shuffled pool)
        idx_r = rsg_pool[rsg_cursor * BATCH_SIZE : (rsg_cursor + 1) * BATCH_SIZE]
        rsg_cursor = (rsg_cursor + 1) % (len(rsg_pool) // BATCH_SIZE)
        idx_r = np.array(idx_r[:BATCH_SIZE])
        rl = train_step_rsg(
            rsg_tr['profile_input_ids'][idx_r],
            rsg_tr['profile_attention_mask'][idx_r],
            rsg_tr['rsg_labels'][idx_r].astype('int32'),
        )
        rsg_ls.append(rl)

    train_ats_mae = float(np.mean(ats_ls))

    # NaN guard
    if np.isnan(train_ats_mae):
        print(f'\nHARD STOP -- NaN train loss at epoch {epoch}')
        model.load_weights(R4_CKPT)
        hard_stop = True
        break

    # Epoch-end validation
    val_mae_100, val_dom_f1, val_dom_ce, _ = eval_ats(ats_vl)
    val_rsg_acc, val_rsg_top3, val_rsg_ce  = eval_rsg(rsg_vl)
    val_loss = combined_val_loss(val_mae_100 / 100.0, val_dom_ce, val_rsg_ce)

    print(
        f'  Ep {epoch:02d}/{MAX_EPOCHS} | '
        f'tr_ATS:{train_ats_mae:.4f}  '
        f'val_MAE:{val_mae_100:.2f}  '
        f'val_DomF1:{val_dom_f1:.3f}  '
        f'val_RSG:{val_rsg_acc:.3f}  '
        f'val_loss:{val_loss:.4f}'
    )

    log_rows.append({
        'epoch':           epoch,
        'train_ats_mae':   round(train_ats_mae, 6),
        'train_dom_loss':  round(float(np.mean(dom_ls)), 6),
        'train_rsg_loss':  round(float(np.mean(rsg_ls)), 6),
        'val_ats_mae_100': round(val_mae_100, 4),
        'val_dom_f1':      round(val_dom_f1,  4),
        'val_rsg_acc':     round(val_rsg_acc, 4),
        'val_rsg_top3':    round(val_rsg_top3, 4),
        'val_combined_loss': round(val_loss, 6),
    })

    # Regression hard stop
    if val_mae_100 > HARD_STOP_MAE_100:
        print(
            f'\n  *** HARD STOP epoch {epoch}: '
            f'val MAE {val_mae_100:.2f} > ceiling {HARD_STOP_MAE_100}. '
            'Restoring best checkpoint. ***'
        )
        model.load_weights(R4_CKPT)
        hard_stop = True
        break

    # Checkpoint on best combined val loss
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = epoch
        patience_counter = 0
        model.save_weights(R4_CKPT)
        print(f'    -> New best val_loss={val_loss:.4f} (MAE={val_mae_100:.2f}) -- checkpoint saved.')
    else:
        patience_counter += 1
        print(f'    -> No improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\n  Early stopping: {PATIENCE} epochs without improvement.')
            model.load_weights(R4_CKPT)
            break

print('\nTraining loop complete.')
print(f'Best epoch: {best_epoch}  |  Best val_loss: {best_val_loss:.4f}')

In [ ]:
# Cell 15: Save training log
if log_rows:
    with open(R4_LOG, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=log_rows[0].keys())
        writer.writeheader()
        writer.writerows(log_rows)
    print(f'Training log saved -> {R4_LOG}')

In [ ]:
# Cell 16: Save final best weights
model.load_weights(R4_CKPT)   # ensure best weights are active
model.save_weights(R4_BEST)
print(f'Final weights saved -> {R4_BEST}')

In [ ]:
# Cell 17: Full test set evaluation
print('Running full evaluation on held-out test set...')

# ATS + Domain test
test_mae_100, test_dom_f1, _, test_per_cls_f1 = eval_ats(ats_tst)

# ATS val (needed for val gate separately)
val_mae_final, val_dom_f1_final, _, val_per_cls_f1 = eval_ats(ats_vl)

# RSG val (spec gate is on val, not test)
rsg_top1_acc, rsg_top3_acc, _ = eval_rsg(rsg_vl)

print(f'\n  ATS val  MAE : {val_mae_final:.2f}')
print(f'  ATS test MAE : {test_mae_100:.2f}')
print(f'  Dom test F1  : {test_dom_f1:.4f}')
print(f'  RSG val Top-1: {rsg_top1_acc:.4f}')
print(f'  RSG val Top-3: {rsg_top3_acc:.4f}')

In [ ]:
# Cell 18: Fresher fairness check (requires merged_final.csv)
fresher_gap    = None
n_fresher      = 0
n_experienced  = 0
fresher_status = 'SKIPPED'

if os.path.exists(MERGED_CSV):
    print(f'Loading merged_final.csv for fresher fairness...')
    import pandas as pd
    df = pd.read_csv(MERGED_CSV)

    # Align with test indices from original data order
    if len(df) > max(ats_tst_idx):
        test_resumes = df.iloc[ats_tst_idx]['resume_text'].fillna('').values
        is_fresher   = np.array([
            any(kw in r.lower() for kw in FRESHER_KEYWORDS)
            for r in test_resumes
        ])
        n_fresher    = int(is_fresher.sum())
        n_experienced = len(is_fresher) - n_fresher

        if n_fresher >= 10:
            # Predict ATS scores on test set
            all_scores = []
            n_tst = len(ats_tst_idx)
            for s in range(0, n_tst, BATCH_SIZE):
                e = min(s + BATCH_SIZE, n_tst)
                ap, _, _ = model(
                    [ats_tst['resume_input_ids'][s:e],
                     ats_tst['resume_attention_mask'][s:e],
                     ats_tst['jd_input_ids'][s:e],
                     ats_tst['jd_attention_mask'][s:e]],
                    training=False,
                )
                all_scores.append(ap.numpy().flatten())
            all_scores = np.concatenate(all_scores) * 100.0

            exp_mean     = float(all_scores[~is_fresher].mean())
            fresh_mean   = float(all_scores[is_fresher].mean())
            fresher_gap  = exp_mean - fresh_mean
            fresher_status = 'PASS' if fresher_gap <= GATE_FRESHER_GAP else 'FAIL'
            print(f'  Fresher samples : {n_fresher}')
            print(f'  Experienced     : {n_experienced}')
            print(f'  Gap (exp-fresh)  : {fresher_gap:.1f} pts  {fresher_status}')
        else:
            print(f'  Only {n_fresher} fresher samples in test set (<10) -- skipping gap.')
            fresher_status = 'N/A (< 10 fresher samples)'
    else:
        print('  CSV row count mismatch with test indices -- skipping.')
        fresher_status = 'SKIPPED (index mismatch)'
else:
    print('  merged_final.csv not found -- fresher fairness skipped.')
    print('  Upload merged_final.csv to ATS_R4/ on Drive to enable this check.')

In [ ]:
# Cell 19: Gate report
g_ats_val  = val_mae_final     < GATE_ATS_VAL_MAE
g_ats_test = test_mae_100      < GATE_ATS_TEST_MAE
g_dom_mac  = test_dom_f1       > GATE_DOM_F1_MACRO
g_dom_cls  = all(f > GATE_DOM_F1_CLASS for f in test_per_cls_f1)
g_rsg      = rsg_top1_acc      > GATE_RSG_ACC
g_fresh    = (fresher_gap is None) or (fresher_gap <= GATE_FRESHER_GAP)

all_pass   = g_ats_val and g_ats_test and g_dom_mac and g_dom_cls and g_rsg and g_fresh

print('\n' + '=' * 65)
print('  INJECTION-R4 — FINAL GATE RESULTS')
print('=' * 65)
print(f'  ATS val  MAE (0-100): {val_mae_final:.2f}  '
      f'{"PASS" if g_ats_val  else "FAIL"}  (gate < {GATE_ATS_VAL_MAE})')
print(f'  ATS test MAE (0-100): {test_mae_100:.2f}  '
      f'{"PASS" if g_ats_test else "FAIL"}  (gate < {GATE_ATS_TEST_MAE})')
print(f'  Domain F1 macro     : {test_dom_f1:.4f}  '
      f'{"PASS" if g_dom_mac  else "FAIL"}  (gate > {GATE_DOM_F1_MACRO})')
print(f'  RSG val Top-1 Acc   : {rsg_top1_acc:.4f}  '
      f'{"PASS" if g_rsg      else "FAIL"}  (gate > {GATE_RSG_ACC})')
print(f'  RSG val Top-3 Acc   : {rsg_top3_acc:.4f}')
print()
print(f'  Per-domain F1 (gate > {GATE_DOM_F1_CLASS} each):')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, test_per_cls_f1)):
    flag = 'PASS' if f1v > GATE_DOM_F1_CLASS else 'FAIL'
    print(f'    [{i}] {name:12s}: {f1v:.4f}  {flag}')
print()
if fresher_gap is not None:
    print(f'  Fresher gap         : {fresher_gap:.1f} pts  '
          f'{"PASS" if g_fresh else "FAIL"}  (gate <= {GATE_FRESHER_GAP})')
    print(f'    Fresher: {n_fresher}  |  Experienced: {n_experienced}')
else:
    print(f'  Fresher gap         : {fresher_status}')
print()
print(f'  Hard stop triggered : {"YES" if hard_stop else "NO"}')
print(f'  Best epoch          : {best_epoch}')
print(f'  Output weights      : {R4_BEST}')
print('=' * 65)

if all_pass:
    print('\nR4 PASSED ALL GATES — proceed to T1')
else:
    print('\n*** R4 FAILED one or more gates — DO NOT proceed to T1. ***')
    print('    Review metrics above and debug before advancing.')

In [ ]:
# Cell 20: Save r4_final_eval_report.json
report = {
    'stage': 'R4',
    'best_epoch': best_epoch,
    'hard_stop_triggered': hard_stop,
    'gates': {
        'ats_val_mae_100':  {'value': round(val_mae_final, 4),  'threshold': f'< {GATE_ATS_VAL_MAE}',  'pass': bool(g_ats_val)},
        'ats_test_mae_100': {'value': round(test_mae_100, 4),   'threshold': f'< {GATE_ATS_TEST_MAE}', 'pass': bool(g_ats_test)},
        'domain_f1_macro':  {'value': round(test_dom_f1, 4),    'threshold': f'> {GATE_DOM_F1_MACRO}', 'pass': bool(g_dom_mac)},
        'domain_f1_all_classes_pass': bool(g_dom_cls),
        'rsg_val_top1_acc': {'value': round(rsg_top1_acc, 4),   'threshold': f'> {GATE_RSG_ACC}',      'pass': bool(g_rsg)},
        'rsg_val_top3_acc': round(rsg_top3_acc, 4),
        'fresher_gap_pts':  fresher_gap if fresher_gap is not None else fresher_status,
        'fresher_gate_pass': bool(g_fresh),
    },
    'per_domain_f1': {
        DOMAIN_NAMES[i]: {'f1': round(float(test_per_cls_f1[i]), 4),
                          'pass': bool(test_per_cls_f1[i] > GATE_DOM_F1_CLASS)}
        for i in range(7)
    },
    'r3_baseline': {
        'val_ats_mae': round(r3_val_mae, 4),
        'val_dom_f1':  round(r3_dom_f1, 4),
        'val_rsg_acc': round(r3_rsg_acc, 4),
    },
    'training_config': {
        'lr': LR, 'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
        'loss_weights': {'ats': W_ATS, 'domain': W_DOM, 'rsg': W_RSG},
    },
    'all_gates_pass': bool(all_pass),
}

with open(R4_REPORT, 'w') as fh:
    json.dump(report, fh, indent=2)
print(f'Report saved -> {R4_REPORT}')
print(json.dumps(report, indent=2))

In [ ]:
# Cell 21: Download results to local machine
# Weights + report are already on Google Drive (persistent).
# Use this cell to also download directly to your browser.
from google.colab import files

print('Downloading r4_final_weights.h5 ...')
files.download(R4_BEST)

print('Downloading r4_final_eval_report.json ...')
files.download(R4_REPORT)

print('Downloading r4_training_log.csv ...')
files.download(R4_LOG)

print('\nSave r4_final_weights.h5 to:')
print('  model/unified_model/r4_final_weights.h5')
print('Save r4_final_eval_report.json to:')
print('  evaluation/r4_final_eval_report.json')